# R2AI Kaggle Args Runner

Sua duy nhat chuoi `RUN_ARGS` ben duoi, sau do bam **Run All**. Notebook se clone/pull repo, chay pipeline A-Z, va tao `results/submission.zip`.

Cac option nhanh:

- Chac an: bo `--use-dense --use-cross-encoder`.
- Manh hon: giu `--use-dense --use-cross-encoder`.
- Chia shard: dat `--num-shards 4 --shard-id 0`, roi chay lai voi shard-id 1, 2, 3; cuoi cung chay `--merge-shards`.


In [ ]:
# Sua args o day. Khong can sua cac cell ben duoi.
RUN_ARGS = """
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--install-deps
--import-docs 5000
--variant balanced
--use-dense
--use-cross-encoder
--num-shards 1
--shard-id 0
--log-every 10
--checkpoint-every 10
"""


In [ ]:
from pathlib import Path
import argparse
import os
import shlex
import subprocess
import sys


def run_cmd(cmd, *, label=""):
    print("\n$", " ".join(str(part) for part in cmd), flush=True)
    try:
        result = subprocess.run([str(part) for part in cmd], check=False)
    except FileNotFoundError as exc:
        print(f"WARNING: {label or cmd[0]} could not start: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"WARNING: {label or cmd[0]} exited with code {result.returncode}", flush=True)
        return False
    return True

parser = argparse.ArgumentParser(add_help=False)
parser.add_argument("--repo-url", default="https://github.com/danhyoyo/r2ai_andgate.git")
parser.add_argument("--branch", default="main")
parser.add_argument("--repo-dir", default="/kaggle/working/r2ai_andgate")
clone_args, runner_args = parser.parse_known_args(shlex.split(RUN_ARGS))

repo_dir = Path(clone_args.repo_dir)
if repo_dir.exists() and (repo_dir / ".git").exists():
    run_cmd(["git", "-C", repo_dir, "pull", "--ff-only"], label="git pull")
elif not repo_dir.exists():
    run_cmd(["git", "clone", "--depth", "1", "--branch", clone_args.branch, clone_args.repo_url, repo_dir], label="git clone")

if not repo_dir.exists():
    print("Repo directory is missing. Enable Internet or upload the repo as a Kaggle Dataset, then rerun.")
else:
    os.chdir(repo_dir)
    print("Repo:", Path.cwd())
    run_cmd(["git", "rev-parse", "--short", "HEAD"], label="git rev-parse")
    run_cmd([sys.executable, "kaggle_runner.py", *runner_args], label="kaggle runner")


## Mau RUN_ARGS

### 1. Chac an nhat, it loi nhat

```text
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--install-deps
--import-docs 5000
--variant balanced
--num-shards 1
```

### 2. Khuyen nghi tren Kaggle GPU

```text
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--install-deps
--import-docs 5000
--variant balanced
--use-dense
--use-cross-encoder
--num-shards 1
```

### 3. Chia 4 shard de tranh timeout

Chay lan luot `--shard-id 0`, `1`, `2`, `3`:

```text
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--install-deps
--import-docs 5000
--variant balanced
--use-dense
--use-cross-encoder
--num-shards 4
--shard-id 0
```

Sau khi co du cac file `results_shard_*_of_4.json`, merge:

```text
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--merge-shards
```

### 4. Corpus lon hon

```text
--repo-url https://github.com/danhyoyo/r2ai_andgate.git
--branch main
--install-deps
--import-docs 20000
--variant recall
--use-dense
--use-cross-encoder
```
